In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
%pip install pyspark

In [32]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

# Create and config Spark
config = SparkConf().setAppName("bai03").setMaster("local[*]")
sparkContext = SparkContext.getOrCreate(conf=config)
sparkSession = SparkSession.builder.appName("bai03").getOrCreate()

In [33]:
data_path = "/content/drive/MyDrive/data/"

# Read file
movies_data = sparkContext.textFile(data_path + "movies.txt")

ratings1_data = sparkContext.textFile(data_path + "ratings_1.txt")
ratings2_data = sparkContext.textFile(data_path + "ratings_2.txt")

users_data = sparkContext.textFile(data_path + "users.txt")

In [34]:
# Parse users.txt
def splition_user(line):
    parts = line.split(',')
    user_id = int(parts[0])
    gender = parts[1]  # M hoặc F
    age = int(parts[2])
    occupation = int(parts[3])
    zipcode = parts[4]
    return (user_id, gender)

users_map = users_data.map(splition_user)

users_dict = users_map.collectAsMap()


In [35]:
def splition_movie(line):
    parts = line.split(',', 2)
    movie_id = int(parts[0])
    title = parts[1]
    genres = parts[2] if len(parts) > 2 else ""
    return (movie_id, title)

movies_map = movies_data.map(splition_movie)

movies_dict = movies_map.collectAsMap() # Dict to retrieve data


In [37]:
# Split ratings: UserID, MovieID, Rating, Timestamp
def splition_rating_with_user(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])
    return (user_id, movie_id, rating)

ratings_1_parsed = ratings1_data.map(splition_rating_with_user)
ratings_2_parsed = ratings2_data.map(splition_rating_with_user)

# Combine 2 file
total_ratings = ratings_1_parsed.union(ratings_2_parsed)

In [38]:
def gender_with_rating(record):
    user_id, movie_id, rating = record
    gender = users_dict.get(user_id, "Unknown Gender")
    return (movie_id, (rating, gender))

ratings_with_gender = total_ratings.map(gender_with_rating)

# Filter genres
filter_ratings_genres = ratings_with_gender.filter(lambda x: x[1][1] in ['M', 'F'])

# Split into F and M
male_ratings = filter_ratings_genres.filter(lambda x: x[1][1] == 'M').map(lambda x: (x[0], x[1][0]))
female_ratings = filter_ratings_genres.filter(lambda x: x[1][1] == 'F').map(lambda x: (x[0], x[1][0]))


In [40]:
# Calculate average point for each movie follow M & F (movie_id, average_rating)
male_analysis = male_ratings.map(lambda x: (x[0], (x[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
male_average_point = male_analysis.map(lambda x: (x[0], x[1][0] / x[1][1]))

female_analysis = female_ratings.map(lambda x: (x[0], (x[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
female_average_point = female_analysis.map(lambda x: (x[0], x[1][0] / x[1][1]))

# Outer join male and female
movie_gender_ratings = male_average_point.fullOuterJoin(female_average_point)

# Add movie name
def add_movie_title(row):
    movie_id, (male_avg_point, female_avg_point) = row
    movie_title = movies_dict.get(movie_id, f"No Name Movie {movie_id}")

    male_avg = f"{male_avg_point:.2f}" if male_avg_point is not None else "NA"
    female_avg = f"{female_avg_point:.2f}" if female_avg_point is not None else "NA"

    return (movie_id, (movie_title, male_avg, female_avg))

results = movie_gender_ratings.map(add_movie_title)

# Collect final data
all_movies = results.collect()

for movie_id, (title, male_avg_display, female_avg_display) in all_movies:
    print(f"{title} - Male_Avg: {male_avg_display}, Female_Avg: {female_avg_display}")

Gladiator (2000) - Male_Avg: 3.59, Female_Avg: 3.64
The Terminator (1984) - Male_Avg: 3.93, Female_Avg: 4.14
Lawrence of Arabia (1962) - Male_Avg: 3.55, Female_Avg: 3.31
Mad Max: Fury Road (2015) - Male_Avg: 4.00, Female_Avg: 3.32
No Country for Old Men (2007) - Male_Avg: 3.92, Female_Avg: 3.83
E.T. the Extra-Terrestrial (1982) - Male_Avg: 3.81, Female_Avg: 3.55
Fight Club (1999) - Male_Avg: 3.50, Female_Avg: 3.50
Psycho (1960) - Male_Avg: NA, Female_Avg: 4.00
The Lord of the Rings: The Fellowship of the Ring (2001) - Male_Avg: 4.00, Female_Avg: 3.80
The Godfather: Part II (1974) - Male_Avg: 4.06, Female_Avg: 3.94
The Silence of the Lambs (1991) - Male_Avg: 3.33, Female_Avg: 3.00
Sunset Boulevard (1950) - Male_Avg: 4.33, Female_Avg: 4.50
The Social Network (2010) - Male_Avg: 4.00, Female_Avg: 3.67
The Lord of the Rings: The Return of the King (2003) - Male_Avg: 3.75, Female_Avg: 3.90
